In [1]:
# requires cell_annotation environment
import sys
import os
import io
import json
import importlib
import numpy as np
import collections
import scipy
import sklearn
from pySankey.sankey import sankey # move to plot_utils in future

import seaborn as sns
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.gridspec import GridSpec

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

In [2]:
# reading in util functions:
# notebook directory
current_dir = os.getcwd()

# project directory
root_dir = os.path.abspath(os.path.join(current_dir, '..', '..'))
os.chdir(root_dir)

# for importing utils
sys.path.append(os.path.join(root_dir, 'src', 'functions'))

import annotation_utils
import anno_class
import core_class_test
import classifier_class
import plot_utils

In [3]:
n12_celesta = pd.read_csv('celesta_n12_predicted.csv') # from independent R script
n12_results = pd.read_csv('n12_results.csv') # from 50/50 train test split

c7_celesta = pd.read_csv('celesta_c7_predicted.csv') # from independent R script
c7_results = pd.read_csv('c7_results.csv') # from 50/50 train test split

In [4]:
n12_annotations = n12_results.loc[n12_results['used_in_training'],:]

c7_annotations = c7_results.loc[c7_results['used_in_training'],:]

# Core N12

In [5]:
n12_annotations.set_index('Object.ID', inplace = True)
n12_celesta.set_index('id', inplace = True)

n12_annotations.index.name = 'id'
n12_celesta.index.name = 'id'

In [6]:
n12_celesta.loc[n12_celesta['predicted'] == 'NKcells','predicted'] = 'NK'
n12_celesta.loc[n12_celesta['predicted'] == 'epithelia','predicted'] = 'Epi'
n12_celesta.loc[n12_celesta['predicted'] == 'myeloid','predicted'] = 'Myeloid'
n12_celesta.loc[n12_celesta['predicted'] == 'Bcells','predicted'] = 'Bcell'
n12_celesta.loc[n12_celesta['predicted'] == 'endothelia','predicted'] = 'Endothelia'
n12_celesta.loc[n12_celesta['predicted'] == 'stroma','predicted'] = 'Stroma'

In [7]:
n12_celesta['predicted'].unique()

array(['Unknown', 'NK', 'Epi', 'Endothelia', 'T_cells', 'CD4_T', 'CD8_T',
       'immune', 'Myeloid', 'Treg', 'Stroma', 'Bcell'], dtype=object)

In [8]:
merged = pd.merge(n12_annotations, n12_celesta, how = 'left', left_index = True, right_index = True)

In [9]:
metrics_list = []
for celltype in n12_annotations['annotations'].unique():
    y_val = merged['annotations']
    y_pred = merged['predicted']
    cls = celltype

    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, labels = [cls], average = 'macro', zero_division = 0)
    precision = precision_score(y_val, y_pred, labels = [cls], average = 'macro', zero_division = 0)
    recall = recall_score(y_val, y_pred, labels = [celltype], average = 'macro', zero_division = 0)
    
    metrics_list.append({'class': cls,
                        'f1': f1,
                        'precision': precision,
                        'recall': recall,
                        'accuracy': accuracy})
                    
    metrics_df = pd.DataFrame(metrics_list)


In [11]:
print('N12')
metrics_df.head

N12


<bound method NDFrame.head of         class        f1  precision    recall  accuracy
0         Epi  0.790698   0.739130  0.850000  0.605634
1       CD4_T  0.666667   0.888889  0.533333  0.605634
2      Stroma  0.731707   0.714286  0.750000  0.605634
3     Myeloid  0.594595   0.647059  0.550000  0.605634
4       Bcell  0.200000   0.333333  0.142857  0.605634
5        Treg  0.727273   0.923077  0.600000  0.605634
6       CD8_T  0.611111   0.687500  0.550000  0.605634
7  Endothelia  0.666667   0.846154  0.550000  0.605634>

In [31]:
metrics = ['f1', 'precision', 'recall']

# Create a figure with 3 subplots arranged in 3 rows and 1 column
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(8, 3))

# Use the tab20 palette to ensure consistent colors across plots
palette = sns.color_palette("tab20", n_colors=len(metrics_df['class'].unique()))
color_map = {cls: palette[i] for i, cls in enumerate(metrics_df['class'].unique())}

# Iterate through each metric and its corresponding subplot
for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Create the bar plot for the current metric
    sns.barplot(
        data=metrics_df,
        x='class',
        y=metric,
        ax=ax,
        alpha = 0.8,
        edgecolor = 'black',
        palette=[color_map[cls] for cls in metrics_df['class']]
    )
    
    # Set the title and labels for the subplot
    ax.set_title(f'{metric.capitalize()}', fontsize=14)
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel(f'Value', fontsize=12)
    ax.set_ylim(0, 1.0) # Ensure the y-axis is consistent for all plots
    ax.tick_params(axis='x', rotation=45)
    ax.set_xticklabels(ax.get_xticklabels(), ha='right')
    ax.grid(False)

# Adjust layout to prevent titles and labels from overlapping
plt.tight_layout(pad=1.0)

# Display the plot
plt.show()

/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:31: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), ha='right')
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:31: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. afte

In [32]:
column1 = merged['annotations']
column2 = merged['predicted']
from scipy.optimize import linear_sum_assignment # for diagnolization in contingency plot


In [33]:
# crosstab for contingency then osrt
crosstab = pd.crosstab(column1, column2, dropna = False)

plt.figure(figsize = (5,5))
sns.heatmap(crosstab,
                annot = True,
                fmt = "d",
                cmap = "viridis",
                cbar = True,
                linewidths = .5,
                linecolor = 'lightgray',
                alpha = 0.7
               )
plt.title('Celetsa vs annotated (N12)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


# Core C7

In [34]:
c7_annotations.set_index('Object.ID', inplace = True)
c7_celesta.set_index('id', inplace = True)

c7_annotations.index.name = 'id'
c7_celesta.index.name = 'id'

In [35]:
c7_celesta.loc[c7_celesta['predicted'] == 'NKcells','predicted'] = 'NK'
c7_celesta.loc[c7_celesta['predicted'] == 'epithelia','predicted'] = 'Epi'
c7_celesta.loc[c7_celesta['predicted'] == 'myeloid','predicted'] = 'Myeloid'
c7_celesta.loc[c7_celesta['predicted'] == 'Bcells','predicted'] = 'Bcell'
c7_celesta.loc[c7_celesta['predicted'] == 'endothelia','predicted'] = 'Endothelia'
c7_celesta.loc[c7_celesta['predicted'] == 'stroma','predicted'] = 'Stroma'

In [36]:
c7_celesta['predicted'].unique()

array(['Epi', 'Stroma', 'CD8_T', 'Treg', 'Endothelia', 'Unknown',
       'immune', 'NK', 'Myeloid', 'Bcell', 'CD4_T', 'T_cells'],
      dtype=object)

In [37]:
merged = pd.merge(c7_annotations, c7_celesta, how = 'left', left_index = True, right_index = True)

In [38]:
metrics_list = []
for celltype in c7_annotations['annotations'].unique():
    y_val = merged['annotations']
    y_pred = merged['predicted']
    cls = celltype

    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, labels = [cls], average = 'macro', zero_division = 0)
    precision = precision_score(y_val, y_pred, labels = [cls], average = 'macro', zero_division = 0)
    recall = recall_score(y_val, y_pred, labels = [celltype], average = 'macro', zero_division = 0)
    
    metrics_list.append({'class': cls,
                        'f1': f1,
                        'precision': precision,
                        'recall': recall,
                        'accuracy': accuracy})
                    
    metrics_df = pd.DataFrame(metrics_list)


In [39]:
print('C7')
metrics_df

C7


,class,f1,precision,recall,accuracy
0,Myeloid,0.611111,0.687500,0.55,0.585185
1,Stroma,0.727273,0.923077,0.60,0.585185
2,Epi,0.627451,0.516129,0.80,0.585185
3,CD4_T,0.250000,0.750000,0.15,0.585185
4,Endothelia,0.871795,0.894737,0.85,0.585185
5,Treg,0.615385,0.500000,0.80,0.585185
6,CD8_T,0.551724,0.888889,0.40,0.585185


In [40]:
metrics = ['f1', 'precision', 'recall']

# Create a figure with 3 subplots arranged in 3 rows and 1 column
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(8, 3))

# Use the tab20 palette to ensure consistent colors across plots
palette = sns.color_palette("tab20", n_colors=len(metrics_df['class'].unique()))
color_map = {cls: palette[i] for i, cls in enumerate(metrics_df['class'].unique())}

# Iterate through each metric and its corresponding subplot
for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Create the bar plot for the current metric
    sns.barplot(
        data=metrics_df,
        x='class',
        y=metric,
        ax=ax,
        alpha = 0.8,
        edgecolor = 'black',
        palette=[color_map[cls] for cls in metrics_df['class']]
    )
    
    # Set the title and labels for the subplot
    ax.set_title(f'{metric.capitalize()}', fontsize=14)
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel(f'Value', fontsize=12)
    ax.set_ylim(0, 1.0) # Ensure the y-axis is consistent for all plots
    ax.tick_params(axis='x', rotation=45)
    ax.set_xticklabels(ax.get_xticklabels(), ha='right')
    ax.grid(False)

# Adjust layout to prevent titles and labels from overlapping
plt.tight_layout(pad=1.0)

# Display the plot
plt.show()

/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:31: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), ha='right')
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_81490/1727869997.py:31: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. afte

In [19]:
column1 = merged['annotations']
column2 = merged['predicted']


In [20]:
# crosstab for contingency then osrt
crosstab = pd.crosstab(column1, column2, dropna = False)

plt.figure(figsize = (5,5))
sns.heatmap(crosstab,
                annot = True,
                fmt = "d",
                cmap = "viridis",
                cbar = True,
                linewidths = .5,
                linecolor = 'lightgray',
                alpha = 0.7
               )
plt.title('Celetsa vs annotated (C7)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
